# Sistemas inteligentes. Aplicaciones

## Práctica 4. Autocorrector

En esta práctica vas a implementar un autocorrector. Este sistema es capaz de detectar aquellas palabras que no están escritas correctamente, y reemplaza cada una de ellas por la palabra más probable que esté a una menor distancia de edición (siempre que esta distancia sea menor o igual a 2).

Posteriormente, deberás crear otra función que, dada una frase y su versión corregida, sea capaz de identificar todas las operaciones necesarias de edición para corregirla.

### Creación del diccionario de probabilidades de las palabras del corpus

Vas a comenzar creando el diccionario que necesitas a partir del corpus de entrada. En este caso, el corpus está formado por 21 libros escritos por Pío Baroja (están almacenados en la carpeta libros). Para leer todos ellos, recuerda que a través de la librería ```os``` puedes obtener una lista con todos sus nombres. Los libros están codificados mediante UTF-8, por lo que puedes leerlos con la siguiente instrucción: ```open('nombreArchivo','r',encoding='UTF-8')```.

A partir de ese corpus debes crear un diccionario con todas las palabras que existen en el vocabulario indicando, para cada una de ellas, pa probabilidad de aparición de esa palabra en el corpus. Como solo queremos trabajar con las palabras, debes eliminar del texto todos aquellos símbolos que no sean considerados palabras por una expresión regular (recuerda los elementos ```\w``` y ```\W``` de las expresiones regulares).

In [ ]:
import os
import re
import numpy as np

word_vocabulary = {}
char_set = set()
books = os.listdir("libros")
for book in books:
    path = "libros/" + book
    file = open(path, 'r', encoding='UTF-8')
    initial_text = file.read()
    cleaned_text = re.sub('[^\w\s]', '', initial_text).lower().split()
    for word in cleaned_text:
        for letter in word:
            if letter not in char_set and letter not in "1234567890":
                char_set.add(letter)
        if word not in word_vocabulary:
            word_vocabulary[word] = 0
        
        word_vocabulary[word] += 1


total_words = sum(word_vocabulary.values())
for word in word_vocabulary:
    word_vocabulary[word] = word_vocabulary[word]/total_words

In [ ]:
word_vocabulary

### Funciones para aplicar una operación de edición
Ahora debes crear 4 funciones, cada una relacionada con una posible operación de edición: insertar, borrar, reemplazar e intercambiar. Cada una de estas funciones debe recibir una palabra y devolver una lista con todos los strings que se puedan obtener al aplicar esa operación de edición sobre la palabra recibida.

In [ ]:
def insert(word, char_set):
    strings = []
    for char in char_set:
        for i in range(len(word)+1):
            strings.append(word[0:i] + char + word[i: len(word) + 1])
    return strings

In [ ]:
def delete(word):
    strings = []
    for i in range(len(word)):
        strings.append(word[0:i] + word[i+1: len(word) + 1])
    return strings


In [ ]:
def replace(word, char_set):
    strings = []
    for char in char_set:
        for i in range(len(word)):
            strings.append(word[0:i] + char + word[i+1: len(word)])

    return strings

In [ ]:
def swap(word):
    strings = []
    for i in range(len(word)-1):
        first_part = word[0:i]
        second_part = word[i+1] + word[i]
        last_part = word[i+2:len(word)]
        string = first_part + second_part + last_part
        strings.append(string)
    
    return strings

### Función una_edicion
Utilizando las 4 funciones anteriores, crea una nueva función que, recibiendo una palabra, devuelva todos lo strings que se encuentran a distancia 1 de edición de dicha palabra.

Además de la palabra, esta función debe recibir como argumento el parámetro *intercambiar*, que por defecto vale *False*. Si este parámetro es verdadero, utilizamos las 4 operaciones de edición (insertar, borrar, reemplazar e intercambiar). Si este parámetro es falso, entonces solo utilizamos como operaciones de edición insertar, borrar y reemplazar.

Ten en cuenta que al aplicar dos operaciones de edición por separado, podríamos obtener un mismo string. Debes asegurarte de que en los strings que devuelve tu función no hay ninguno repetido.

In [ ]:
def one_edition(word, char_set, swap_bool = False):
    strings = []
    strings += (insert(word, char_set))
    strings += (delete(word))
    strings += (replace(word, char_set))

    if swap_bool:
        strings += (swap(word))

    return set(strings)

### Función dos_ediciones

Crea una función que recibe una palabra y devuelve todos los strings que se encuentran a 2 unidades de distancia de edición de esa palabra. Al igual que en el caso anterior, utiliza un pa´rametro intercambiar para permitir usar o no este tipo de operación de edición.

Vuelve a tener en cuenta que no debes devolver strings repetidos, aunque hayan sido creados aplicando diferentes operaciones de edición.

In [ ]:
def two_editions(word, char_set, swap_bool = False):
    strings = []
    one_edition_words = one_edition(word, char_set, swap_bool)
    for edited_word in one_edition_words:
        strings += (insert(edited_word, char_set))
        strings += (delete(edited_word))
        strings += (replace(edited_word, char_set))

        if swap_bool:
            strings += (swap(edited_word))

    return set(strings)


### Función corregir

Utilizando las funciones anteriores, crea una nueva función que recibe una palabra y devuelve la o las correcciones sugeridas. Esta función recibe como argumentos la palabra a corregir, el diccionario de probabilidades de cada palabra del corpus, un valor entero (n) que indica el número máximo de correcciones sugeridas a devolver y un valor booleano que indica si queremos utilizar la operación de edición intercambiar o no.

Según la palabra recibida, esta función debe devolver:
* si la palabra existe en el diccionario, la devuelve tal cual
* si no
    * si existen palabras reales a distancia una, devuelve las n más probables, en orden de mayor a menor
    * si no
           * si existen palabras reales a distancia dos, devuelve las n más probables, en orden de mayor a menor
           * si no, devuelve la palabra original

El parámetro del número máximo de palabras a devolver se utiliza solo entre palabras que estén a la misma distancia de edición. Por ejemplo, si n=10 y existen 2 palabras a distancia de edición 1 y 20 palabras a distancia de edición 2, la función solo devolverá las dos palabras a distancia 1. Si, en otro caso, n=10 y existen 15 palabras a distancia de edición 1 y 30 palabras a distancia de edición 2, la función devolverá las 10 palabras más probables que están a distancia de edición 1.

In [ ]:
def correct(word, word_vocabulary, n, char_set, swap_bool):
    if word not in word_vocabulary:
        strings = []
        one_edition_words = one_edition(word, char_set, swap_bool)
        for edited_word in one_edition_words:
            if edited_word in word_vocabulary:
                strings.append(edited_word)

        if len(strings) != 0: #si hay palabras a distancia = 1
            probabilities = np.zeros(len(strings))
            for i, accepted_word in enumerate(strings):
                probabilities[i] = word_vocabulary[accepted_word]

            indices = np.argsort(-probabilities)
            return list(np.array(strings)[indices][:n])

        else: #si no hay palabras a distancia = 1, miramos las de distancia = 2
            two_editions_words = two_editions(word, char_set, swap_bool)
            for edited_word in two_editions_words:
                if edited_word in word_vocabulary:
                    strings.append(edited_word)
                
            if len(strings) != 0: #si hay palabras a distancia = 2
                probabilities = np.zeros(len(strings))
                for i, accepted_word in enumerate(strings):
                    probabilities[i] = word_vocabulary[accepted_word]

                indices = np.argsort(-probabilities)
                return list(np.array(strings)[indices][:n])

            return word


    else:
        return word

### Ejecución

Ejecuta las funciones anteriores para, leída una palabra del usuario, devuelva las (máximo 5) palabras reales más probables a menor distancia de edición.

In [ ]:
word = input("Escribe una palabra: ")
words = correct(word, word_vocabulary, 5, char_set, True)
if isinstance(words, str):
    print("Sin corrección")
else:
    print("Palabras sugeridas:", *words)

### Corrección de una frase

En este caso vamos a trabajar con una frase completa. Para cada una de las palabras que no sean correctas, debes cambiarlas por la palabra que esté a menor distancia de edición que sea más probable. Una vez que tienes la frase corregida, llama a otra función que calcule el número mínimo de operaciones de edición necesarias para pasar de la frase original a la corregida.

En este caso, fijamos que no se puede utilizar la operación de intercambiar y los costes de las otras tres operaciones los pasamos como parámetro a la función que calcula la distancia de edición.

Para calcular la distancia de edición mínima, utiliza el algoritmo de programación dinámica, y devuelve tanto la distancia mínima como la matriz calculada.

In [ ]:
def minimum_distance(phrase1, phrase2, insert_cost, delete_cost, replace_cost):
    m = len(phrase1)+1
    n = len(phrase2)+1
    matrix = np.zeros((m, n))
    matrix[0] = np.arange(n)*insert_cost
    matrix[:, 0] = np.arange(m)*delete_cost
    for i in range(1, m):
        for j in range(1, n):
            if phrase1[i-1] == phrase2[j-1]:
                matrix[i, j] = matrix[i-1, j-1]
            else:
                matrix[i, j] = min(matrix[i-1, j] + delete_cost, matrix[i, j-1] + insert_cost, matrix[i-1, j-1] + replace_cost)

    """ for fila in matrix:
        print(fila) """
    return matrix



phrase = "hola hoy haze uj bueb dia"
new_phrase = []
for word in phrase.split():
    corrected = correct(word, word_vocabulary, 1, char_set, True)
    if isinstance(corrected, str):
        corrected = [corrected]
    new_phrase += corrected
final = " ".join(new_phrase)
print(final)

minimum_distance(phrase, final, 1, 1, 1)

In [ ]:
def distance_proportion(original: str, corrected: str, correct: str):
    m1 = minimum_distance(corrected, correct, 1, 1, 1)
    m2 = minimum_distance(original, correct, 1, 1, 1)

    d1 = m1[-1, -1]
    d2 = m2[-1, -1]

    if d2 == 0:
        return 0
    return d1 / d2
    
    
original = "preuab"
corregida = "presa"
correcta = "prueba"

dp = distance_proportion(original, corregida, correcta)
print(dp)

### Todas las posibles ediciones
Para finalizar, escribe una función que recibe dos palabras (la original y la corregida) y la matriz de distancias mínimas calculadas mediante el algoritmo de programación dinámica. Esta función debe devolver la lista de ediciones que se debe hacer sobre la palabra original para conseguir la palabra corregida.

Como puede haber varios conjuntos de ediciones que obtengan la palabra corregida con el mísmo número de ediciones, debes devolver todos ellos.

In [ ]:
from typing import Any


def editions(i, d, r):
    editions = []
    editions_map = ["insertar", "borrar", "cambiar"]
    table = np.array([i, d, r])
    sorted_index = np.argsort(table)
    editions.append(editions_map[sorted_index[0]])
    i = 1
    while i < len(sorted_index) and table[sorted_index[0]] == table[sorted_index[i]]:
        editions.append(editions_map[sorted_index[i]])
        i+=1
    return editions

def edition_lists(word1, word2, matrix):

    lista = []
    m = matrix.shape[0]-1
    n = matrix.shape[1]-1

    if matrix[m, n] == 0:
        return [[]]

    # Solo se pueden insertar (word1 agotada)
    if m == 0:
        sub_paths = edition_lists(word1, word2[0:n-1], matrix[0:m+1, 0:n])
        for path in sub_paths:
            lista.append(path + [f"insertar {word2[n-1]}"])
        return lista

    # Solo se pueden borrar (word2 agotada)
    if n == 0:
        sub_paths = edition_lists(word1[0:m-1], word2, matrix[0:m, 0:n+1])
        for path in sub_paths:
            lista.append(path + [f"borrar {word1[m-1]}"])
        return lista
        
    else:
        
        if len(word1) >= 1 and len(word2) >= 1 and word1[m-1] == word2[n-1]:
            #print("NADA")
            edited_word1 = word1[0:m-1]
            edited_word2 = word2[0:n-1]
            paths = edition_lists(edited_word1, edited_word2, matrix[0:m, 0:n])
            return paths

        insert_value = matrix[m, n-1]
        delete_value = matrix[m-1, n]
        replace_value = matrix[m-1, n-1]
        possible_editions = editions(insert_value, delete_value, replace_value)
        #print(possible_editions)
        for edition in possible_editions:
            if edition == "insertar":

                #print(f"insertar {word2[n-1]}")
                edited_word2 = word2[0:n-1]
                subpaths = edition_lists(word1, edited_word2, matrix[0:m+1, 0:n])
                for path in subpaths:
                    lista.append(path + [f"insertar {word2[n-1]}"])

            if edition == "borrar":

                #print(f"borrar {word1[m-1]}")
                edited_word1 = word1[0:m-1]
                subpaths: Any = edition_lists(edited_word1, word2, matrix[0:m, 0:n+1])
                for path in subpaths:
                    lista.append(path + [f"borrar {word1[m-1]}"])

            if edition == "cambiar":
                
                #print(f"cambiar {word1[m-1]} por {word2[n-1]}")
                
                edited_word1 = word1[0:m-1]
                edited_word2 = word2[0:n-1]

                subpaths = edition_lists(edited_word1, edited_word2, matrix[0:m, 0:n])
                for path in subpaths:
                    lista.append(path + [f"cambiar {word1[m-1]} por {word2[n-1]}"])

        return lista

In [ ]:
word1 = "maren"
word2 = "marron"
matrix = minimum_distance(word1, word2, 1, 1, 1)
paths = edition_lists(word1, word2, matrix)
print(f"Para pasar de {word1} a {word2} se pueden seguir los diferentes caminos:")
for path in paths:
    print("-----------------")
    for change in path:
        print(change)

In [ ]:
def edition_lists(word1, word2, matrix, final_editions = []):

    #print("----------------------")
    #print(word1, word2)
    #print(matrix)
    lista = []
    #print(final_editions)
    

    m = matrix.shape[0]-1
    n = matrix.shape[1]-1

    if matrix[m, n] == 0:
        return final_editions
        
    else:
        if word1[m-1] == word2[n-1]:
            #print("NADA")
            edited_word1 = word1[0:m-1]
            edited_word2 = word2[0:n-1]
            return edition_lists(edited_word1, edited_word2, matrix[0:m, 0:n], final_editions)

        insert_value = matrix[m, n-1]
        delete_value = matrix[m-1, n]
        replace_value = matrix[m-1, n-1]
        possible_editions = editions(insert_value, delete_value, replace_value)
        #print(possible_editions)
        for edition in possible_editions:
            if edition == "insertar":

                #print(f"insertar {word2[n-1]}")
                edited_word2 = word2[0:n-1]
                final_editions += [f"insertar {word2[n-1]}"]
                return edition_lists(word1, edited_word2, matrix[0:m+1, 0:n], final_editions)

            if edition == "borrar":

                #print(f"borrar {word1[m-1]}")
                edited_word1 = word1[0:m-1]
                final_editions += [f"borrar {word1[m-1]}"]
                return edition_lists(edited_word1, word2, matrix[0:m, 0:n+1], final_editions)

            if edition == "cambiar":
                #print(f"cambiar {word1[m-1]} por {word2[n-1]}")
                final_editions += [f"cambiar {word1[m-1]} por {word2[n-1]}"]
                edited_word1 = word1[0:m-1]
                edited_word2 = word2[0:n-1]
                return edition_lists(edited_word1, edited_word2, matrix[0:m, 0:n], final_editions)

In [ ]:
import urllib.request
import pandas as pd
import spacy

nlp = spacy.load("es_core_news_sm")
url = "https://raw.githubusercontent.com/unimorph/spa/master/spa"
urllib.request.urlretrieve(url, "spa.tsv")

print("Descargado!")

df = pd.read_csv(
    "spa.tsv",
    sep="\t",
    header=None,
    names=["forma", "etiquetas"],
    comment="#"
)
dictionary = dict(zip(df.iloc[:, 0], df.iloc[:, 1]))

In [ ]:
import stanza
stanza.download('es') 
nlp = stanza.Pipeline(lang='es', processors='tokenize,pos,lemma,depparse')

In [ ]:
def puntuacion(frase):
    lista = frase.split()
    suma = 0
    for i in range(len(lista) - 1):
        w_1 = generate_grammar_block(lista[i], dictionary, nlp)
        w_2 = generate_grammar_block(lista[i+1], dictionary, nlp)
        suma += structures[tuple([w_1, w_2])]
    return suma

In [ ]:
class GrammarModel():
    def __init__(self, dictionary, nlp) -> None:
        self.structures = {}
        self.nlp = nlp
        self.dictionary = dictionary
        
    def generate_grammar_block(self, word):
        try:
            grammar_block = self.dictionary[word]
        except:
            doc = self.nlp(word)
            for token in doc:
                grammar_block = f"{token.pos_}_{token.morph}"

        return grammar_block

    def learn(self, path, n, limit):
        file = open(path, 'r', encoding='UTF-8')
        initial_text = file.readlines()
        for s, sentence in enumerate(initial_text):
            print(f"{s}/{limit}")
            sentence_lowed = sentence.lower().split()
            sentence_lowed = [word for word in sentence_lowed if word != "digito"]
            for i in range(len(sentence_lowed) - n + 1):
                word_list = []
                for j in range(n):
                    word_j = sentence_lowed[i + j]
                    grammar_block_j = self.generate_grammar_block(word_j)
                    word_list.append(grammar_block_j)

                structure = tuple(word_list)

                if structure not in self.structures:
                    self.structures[structure] = 0
                
                self.structures[structure] += 1

            if s == limit:
                break

    def get_structures(self):
        s = dict(sorted(self.structures.items(), key=lambda x: x[1], reverse=True))
        return s

    def show_n_gram_score(self, n_gram_words):

        n_gram_list = []
        for word in n_gram_words:
            n_gram_list.append(self.generate_grammar_block(word))

        n_gram = tuple(n_gram_list)
        if n_gram not in self.structures:
            print("NO HAY INFORMACIÓN. SUMA 0 ... ?")
        else:
            print(self.structures[n_gram])


In [ ]:
grammar_model = GrammarModel(dictionary, nlp)
#estaría interesante almacenar las estructuras mas comunes para distintos valores de n
grammar_model.learn("spanish_billion_words_67", 2, 10000)

In [ ]:
frase = "ayer las vieron"
grammar_model.show_n_gram_score(frase.split())

In [ ]:
grammar_model.generate_grammar_block("vio")